In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import random
from torch.optim.swa_utils import AveragedModel, SWALR
import time
import torch.optim as optim

In [2]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import os
from datetime import datetime

In [3]:
torch.backends.cudnn.benchmark = True

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda")

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
train_data = np.load(r"/content/drive/MyDrive/quickdraw_train.npz")
print(train_data.files)

test_data = np.load(r"/content/drive/MyDrive/quickdraw_test.npz")
print(test_data.files)

['x_train', 'y_train', 'class_names']
['test_images']


In [7]:
X = train_data["x_train"]
y = train_data["y_train"]
class_names = train_data["class_names"]

X_test = test_data["test_images"]

print("Train shape:", X.shape)
print("Labels shape:", y.shape)
print("Test shape:", X_test.shape)
print("Number of classes:", len(class_names))

Train shape: (60000, 784)
Labels shape: (60000,)
Test shape: (15000, 784)
Number of classes: 15


In [8]:
print("Unique labels:", np.unique(y))

Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [9]:
print(X.shape)

(60000, 784)


In [10]:
print(X.dtype)
print(X.min(), X.max())

uint8
0 255


In [11]:
class QuickDrawDataset(Dataset):
    def __init__(self, images, labels=None, augment=False):
        self.images = images.astype(np.float32) / 255.0
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def augment_fn(self, x):
      x = x.reshape(28, 28)
      # Random horizontal flip
      if random.random() > 0.5:
          x = torch.flip(x, dims=[1])
      # Random erasing
      if random.random() > 0.5:
          i, j = random.randint(0, 20), random.randint(0, 20)
          h, w = random.randint(4, 8), random.randint(4, 8)
          x[i:i+h, j:j+w] = 0
      x = x.reshape(784)
      noise = torch.randn_like(x) * 0.03
      return torch.clamp(x + noise, 0, 1)

    def __getitem__(self, idx):
        x = torch.tensor(self.images[idx])

        if self.augment:
            x = self.augment_fn(x)

        if self.labels is not None:
            y = torch.tensor(self.labels[idx]).long()
            return x, y

        return x

In [12]:
class ChampionMLP(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes, dropout):
        super().__init__()

        layers = []
        prev = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [13]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [14]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Mixup
        mixed_x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        outputs = model(mixed_x)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        optimizer.step()

        # For accuracy logging, use original x (not mixed)
        with torch.no_grad():
            clean_out = model(x)
        preds = torch.argmax(clean_out, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds = torch.argmax(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

In [15]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
# ── Log file setup ────────────────────────────────────────────
LOG_FILE = f"train_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def log(msg):
    print(msg)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

log(f"🗂 Log file: {LOG_FILE}")
log(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# ─────────────────────────────────────────────────────────────

# ── Fixed hyperparameters from your selection ─────────────────
hidden_sizes = [896, 654, 477, 349, 254, 186]
dropout      = 0.22838018994159953
lr           = 0.001733336434313803
wd           = 1.3321907293603369e-05

EPOCHS       = 40
SWA_START    = 25
SWA_LR       = lr * 0.3

# Optional: deterministic seeds (comment out if you don't need exact repeatability)
# import random
# torch.manual_seed(42); np.random.seed(42); random.seed(42)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

# ── Bookkeeping ───────────────────────────────────────────────
log("\n" + "="*60)
log("🏁 FINAL RUN: Train on FULL train set (no CV) + Predict test")
log("="*60)
log(f"Architecture: {hidden_sizes}")
log(f"Dropout: {dropout:.6f} | LR: {lr:.12f} | WD: {wd:.12f}")
log(f"SWA_START: {SWA_START} | SWA_LR: {SWA_LR:.6f}")
log(f"EPOCHS: {EPOCHS}")

# ── Data (full-train only; test has no labels) ────────────────
BATCH_TRAIN = 512
BATCH_TEST  = 512

train_ds   = QuickDrawDataset(X, y, augment=True)
train_dl   = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True)
# clean (no-augment) loader for SWA BN update at the end
train_clean_ds = QuickDrawDataset(X, y, augment=False)
train_clean_dl = DataLoader(train_clean_ds, batch_size=BATCH_TRAIN, shuffle=False)

test_ds    = QuickDrawDataset(X_test, labels=None, augment=False)
test_dl    = DataLoader(test_ds, batch_size=BATCH_TEST, shuffle=False)

# ── Model ─────────────────────────────────────────────────────
model = ChampionMLP(784, len(class_names), hidden_sizes, dropout).to(device)

# Parameter budget check
param_count = count_parameters(model)
log(f"Parameters: {param_count:,}")
if param_count > 3_000_000:
    raise AssertionError(f"❌ Exceeds 3M parameter limit — Aborting training. Got: {param_count:,}")

# ── Optimizer / Schedulers / Loss ─────────────────────────────
optimizer     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
scheduler     = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
swa_model     = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
criterion     = nn.CrossEntropyLoss(label_smoothing=0.05)

best_train = 0.0
for epoch in range(EPOCHS):
    # Train one epoch on FULL train
    train_acc = train_epoch(model, train_dl, optimizer, criterion)

    # Scheduler / SWA phase
    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
        tag = " [SWA]"
    else:
        scheduler.step()
        tag = ""

    best_train = max(best_train, train_acc)
    log(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {train_acc:.4f}{tag}")

# Finalize BN for SWA and select final weights
from torch.optim.swa_utils import update_bn
update_bn(train_clean_dl, swa_model, device=device)
final_model = swa_model
final_model.eval()

# ── Save final full-train weights ─────────────────────────────
os.makedirs("checkpoints", exist_ok=True)
final_ckpt = "checkpoints/champion_fulltrain_final.pt"
torch.save({
    "epoch": EPOCHS,
    "val_acc": None,              # no validation in final run
    "arch": hidden_sizes,
    "dropout": dropout,
    "lr": lr,
    "wd": wd,
    "params": param_count,
    "state_dict": (final_model.module if hasattr(final_model, "module") else final_model).state_dict(),
    "used_swa": True
}, final_ckpt)
log(f"💾 Saved final weights: {final_ckpt}")

# ── Inference on test set (no labels) ─────────────────────────
all_preds = []
with torch.no_grad():
    for xb in test_dl:
        xb = xb.to(device)
        logits = final_model(xb)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy())
import numpy as np
all_preds = np.concatenate(all_preds)  # shape: (N_test,)

# Save predictions (ready to paste on portal)
os.makedirs("final_leaderboard_run", exist_ok=True)
pred_npy = "final_leaderboard_run/test_predictions.npy"
pred_csv = "final_leaderboard_run/test_predictions.csv"  # single-row CSV
np.save(pred_npy, all_preds)
np.savetxt(pred_csv, all_preds.reshape(1, -1), fmt="%d", delimiter=",")

log(f"Predictions shape: {all_preds.shape} | dtype: {all_preds.dtype}")
log(f"Epochs: {EPOCHS} | Param Count: {param_count:,}")
log(f"🔖 Copy from: {pred_csv}")
print(",".join(map(str, all_preds)))

🗂 Log file: train_log_20260302_111520.txt
Started at: 2026-03-02 11:15:20

🏁 FINAL RUN: Train on FULL train set (no CV) + Predict test
Architecture: [896, 654, 477, 349, 254, 186]
Dropout: 0.228380 | LR: 0.001733336434 | WD: 0.000013321907
SWA_START: 25 | SWA_LR: 0.000520
EPOCHS: 40
Parameters: 1,914,022
Epoch 01/40 | Train Acc: 0.6175
Epoch 02/40 | Train Acc: 0.6957
Epoch 03/40 | Train Acc: 0.7281
Epoch 04/40 | Train Acc: 0.7480
Epoch 05/40 | Train Acc: 0.7648
Epoch 06/40 | Train Acc: 0.7779
Epoch 07/40 | Train Acc: 0.7898
Epoch 08/40 | Train Acc: 0.7970
Epoch 09/40 | Train Acc: 0.8047
Epoch 10/40 | Train Acc: 0.8095
Epoch 11/40 | Train Acc: 0.7862
Epoch 12/40 | Train Acc: 0.7931
Epoch 13/40 | Train Acc: 0.8004
Epoch 14/40 | Train Acc: 0.8085
Epoch 15/40 | Train Acc: 0.8156
Epoch 16/40 | Train Acc: 0.8219
Epoch 17/40 | Train Acc: 0.8281
Epoch 18/40 | Train Acc: 0.8351
Epoch 19/40 | Train Acc: 0.8404
Epoch 20/40 | Train Acc: 0.8492
Epoch 21/40 | Train Acc: 0.8519
Epoch 22/40 | Train Ac